In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 
import os
import numpy as np

In [2]:
df = pd.read_csv("/Users/sudhanvabharadwaj/Downloads/IMC_Prosperity4/Cleaned_Data/emerald_run_rows.csv")

In [3]:
df

,timestamp,product,best_bid,best_ask,best_bid_volume,best_ask_volume,fair,prev_pos,curr_pos,pos_delta,...,tick_sell_fills,cum_buy_fills,cum_sell_fills,buy_quote,sell_quote,buy_order_qty,sell_order_qty,buy_capacity_after,sell_capacity_after,own_trades_seen_count
0,0,EMERALDS,9992,10008,15,-15,10000,0,0,0,...,0,0,0,9997,10003,40,40,40,40,0
1,100,EMERALDS,9992,10008,13,-13,10000,0,0,0,...,0,0,0,9997,10003,40,40,40,40,0
2,200,EMERALDS,9992,10008,13,-13,10000,0,0,0,...,0,0,0,9997,10003,40,40,40,40,0
3,300,EMERALDS,9992,10008,11,-11,10000,0,0,0,...,0,0,0,9997,10003,40,40,40,40,0
4,400,EMERALDS,9992,10008,13,-13,10000,0,0,0,...,0,0,0,9997,10003,40,40,40,40,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,199500,EMERALDS,9992,10008,11,-11,10000,-16,-16,0,...,0,0,6,9997,10003,40,40,56,24,1
1996,199600,EMERALDS,9992,10008,11,-11,10000,-16,-16,0,...,0,0,6,9997,10003,40,40,56,24,1
1997,199700,EMERALDS,9992,10008,14,-14,10000,-16,-16,0,...,0,0,6,9997,10003,40,40,56,24,1
1998,199800,EMERALDS,9992,10008,15,-15,10000,-16,-16,0,...,0,0,6,9997,10003,40,40,56,24,1


In [4]:
df.columns

Index(['timestamp', 'product', 'best_bid', 'best_ask', 'best_bid_volume',
       'best_ask_volume', 'fair', 'prev_pos', 'curr_pos', 'pos_delta',
       'tick_buy_fills', 'tick_sell_fills', 'cum_buy_fills', 'cum_sell_fills',
       'buy_quote', 'sell_quote', 'buy_order_qty', 'sell_order_qty',
       'buy_capacity_after', 'sell_capacity_after', 'own_trades_seen_count'],
      dtype='object')

In [5]:
def prepare_data(df):
    # Ensure chronological order
    df = df.sort_values('timestamp')
    # Calculate Mid-Price for future edge calculations
    df['mid'] = (df['best_bid'] + df['best_ask']) / 2
    return df

df = prepare_data(df)

In [ ]:
def analyze_passive_fill_probability(df, horizons=[1, 5, 10]):
    results = []
    
    # We test offsets relative to the 'Wall' (The Best Bid/Ask)
    # Offset +1 means jumping in front of the wall by 1 tick
    offsets = [0, 1, 2, 3] 
    
    for offset in offsets:
        for h in horizons:
            # HYPOTHETICAL BID: wall + offset
            # We get filled if a SELLER hits the bid. 
            # This happens if the FUTURE 'best_bid' drops BELOW our price, 
            # OR if the FUTURE 'best_ask' touches our price.
            
            my_bid_price = df['best_bid'] + offset
            my_ask_price = df['best_ask'] - offset
            
            # Look ahead H ticks to find the LOWEST best_bid (to see if our bid was hit)
            # and the HIGHEST best_ask (to see if our ask was hit)
            future_low_bid = df['best_bid'].shift(-h).rolling(window=h).min()
            future_high_ask = df['best_ask'].shift(-h).rolling(window=h).max()
            
            # A passive BID is filled if the market moves down to it or through it
            bid_filled = my_bid_price >= future_low_bid
            
            # A passive ASK is filled if the market moves up to it or through it
            ask_filled = my_ask_price <= future_high_ask
            
            results.append({
                'offset_from_wall': offset,
                'horizon': h,
                'bid_fill_prob': bid_filled.mean(),
                'ask_fill_prob': ask_filled.mean()
            })
            
    return pd.DataFrame(results)

fill_surface = analyze_passive_fill_probability(df)
print(fill_surface)

    offset  horizon  bid_fill_prob  ask_fill_prob
0        2        1         0.0000         0.0000
1        2        3         0.0000         0.0000
2        2        5         0.0000         0.0000
3        2       10         0.0020         0.0005
4        2       15         0.0035         0.0010
5        2       25         0.0045         0.0020
6        3        1         0.0000         0.0000
7        3        3         0.0000         0.0000
8        3        5         0.0000         0.0000
9        3       10         0.0020         0.0005
10       3       15         0.0035         0.0010
11       3       25         0.0045         0.0020
12       4        1         0.0000         0.0000
13       4        3         0.0000         0.0000
14       4        5         0.0000         0.0000
15       4       10         0.0020         0.0005
16       4       15         0.0035         0.0010
17       4       25         0.0045         0.0020
18       5        1         0.0000         0.0000


In [23]:
fill_surface_sorted = fill_surface.sort_values(
    by=['bid_fill_prob', 'ask_fill_prob', 'horizon'],
    ascending=[False, False, True]
)

fill_surface_sorted

,offset,horizon,bid_fill_prob,ask_fill_prob
5,2,25,0.0045,0.0020
11,3,25,0.0045,0.0020
17,4,25,0.0045,0.0020
23,5,25,0.0045,0.0020
29,6,25,0.0045,0.0020
35,7,25,0.0045,0.0020
4,2,15,0.0035,0.0010
10,3,15,0.0035,0.0010
16,4,15,0.0035,0.0010
22,5,15,0.0035,0.0010


In [24]:
fill_surface_sorted['score'] = fill_surface_sorted['bid_fill_prob'] + fill_surface_sorted['ask_fill_prob']

fill_surface_sorted['rank'] = (
    fill_surface_sorted.groupby('horizon')['score']
      .rank(method='first', ascending=False)
)


top5 = fill_surface_sorted[fill_surface_sorted['rank'] <= 5].sort_values(['horizon', 'rank'])
top5

,offset,horizon,bid_fill_prob,ask_fill_prob,score,rank
0,2,1,0.0000,0.0000,0.0000,1.0
6,3,1,0.0000,0.0000,0.0000,2.0
12,4,1,0.0000,0.0000,0.0000,3.0
18,5,1,0.0000,0.0000,0.0000,4.0
24,6,1,0.0000,0.0000,0.0000,5.0
1,2,3,0.0000,0.0000,0.0000,1.0
7,3,3,0.0000,0.0000,0.0000,2.0
13,4,3,0.0000,0.0000,0.0000,3.0
19,5,3,0.0000,0.0000,0.0000,4.0
25,6,3,0.0000,0.0000,0.0000,5.0
